# Lab 6 — DATS 6103

**Name:**

This notebook mirrors the lab page. Everything here also runs in the browser at
<https://www.smajhi.com/DATS-6103/labs/pandas-II.html>, where the self-checks and the timers live; work in whichever you
prefer. The discussion half of the lab is not in this file — it happens away
from the keyboard.

**Conditions.** No AI assistants. The official Python, NumPy, Pandas,
Matplotlib and scikit-learn documentation and the course notes may be consulted
freely.

Run the self-check under each answer. It tells you whether your answer is right
without telling you what the answer is.

## Part A—Discussion

**Not submitted, and not written at the keyboard.** Work these in pairs, out loud, one of you holding the rubric card. The rubric cards are on the lab page: <https://www.smajhi.com/DATS-6103/labs/pandas-II.html>.

### A1. Two ways to lose the same people

*Suggested: 5 minutes.*

A survey asks for income. The people with the highest incomes are the likeliest
to leave it blank.

Your colleague offers two fixes: `dropna()` the rows, or `fillna()` the gaps
with the mean of the answers. Agree on what each one does to **the mean** and to
**the standard deviation**, and which of the two is more dangerous in a report.

### A2. The merge that multiplies

*Suggested: 5 minutes.*

`orders` has 5,000 rows, one per order, with a `cust_id`. `lookup` has 1,200
rows, and `cust_id` is supposed to be unique in it---but 40 customers appear
twice.

1. How many rows does `orders.merge(lookup, on="cust_id")` return, if every
   order's customer is in the lookup and the duplicated customers placed 3
   orders each?
2. The merged total spend is now wrong. In which direction, and by how much?
3. Name the one line you would add before and after the merge to catch it, and
   the argument that would have made the merge itself complain.

### A3. What forward filling assumes

*Suggested: 5 minutes.*

Two tables, each with gaps in a `price` column, and the same line of code
applied to both: `df["price"] = df["price"].ffill()`.

- **Table 1**: one stock, one row per trading day, sorted by date.
- **Table 2**: a product catalog sorted by product ID.

Decide for each whether the line is reasonable, and say what the assumption is
in one sentence. Then: the catalog is re-sorted by category and the line is run
again. What changes?

### A4. Which shape, and what `pivot_table` did quietly

*Suggested: 5 minutes.*

You have sales by store and month.

1. Which shape---wide or long---do you want for plotting with a line per store?
   For fitting `sales ~ month + store`? For a slide?
2. `pivot` raises on your data and a colleague swaps in `pivot_table`, which
   works. What happened, and what number are you now looking at?

### A5. A grouped result is a hierarchical index

*Suggested: 5 minutes.*

`sales.groupby(["region", "quarter"])["revenue"].sum()` gives a Series with a
two-level index.

1. What does `.unstack()` return, and how does `.unstack(level=0)` differ from
   `.unstack(level=1)`?
2. What does `.loc["north"]` return, and what happened to the `region` level?
3. Someone suggests `reset_index()` to "get rid of the MultiIndex". What do you
   gain and what do you lose?

## Part B—Build

### B1. Where the holes are

*Suggested: 6 minutes.*

`survey` has 500 respondents and four columns, with gaps in three of them.
Before deciding anything, count.

- `per_column`---a Series of the number of missing values in each column, in the
  frame's column order;
- `complete_rows`---how many rows have no missing value at all;
- `worst`---the name of the column with the most gaps, as a string.

In [ ]:
import numpy as np
import pandas as pd
pd.set_option("display.precision", 2)
rng = np.random.default_rng(6103)
n = 500
income = rng.lognormal(10.8, 0.5, n).round(-2)
survey = pd.DataFrame({
    "resp_id": np.arange(1, n + 1),
    "age": rng.integers(18, 80, n).astype(float),
    "income": income,
    "region": rng.choice(["north", "south", "east", "west"], n),
})
survey.loc[rng.random(n) < 0.08, "age"] = np.nan
survey.loc[rng.random(n) < (income / income.max()) * 0.6, "income"] = np.nan
survey.loc[rng.random(n) < 0.04, "region"] = None

In [ ]:
per_column = ...
complete_rows = ...
worst = ...

print(per_column)
print(complete_rows, "complete rows;", worst, "has the most gaps")

In [ ]:
# self-check: run this after your answer above
assert isinstance(per_column, pd.Series), f"per_column should be a Series, got {type(per_column).__name__}"
assert list(per_column.index) == list(survey.columns), \
    "per_column should be indexed by the column names, in the frame's order"
assert list(per_column) == list(survey.isna().sum()), "per_column does not match the missing counts"
assert int(complete_rows) == int(len(survey.dropna())), \
    f"complete_rows should be the rows with no missing value, {len(survey.dropna())}"
assert isinstance(worst, str) and worst == str(survey.isna().sum().idxmax()), \
    "worst should be the column name with the most missing values, as a string"
print("looks right")

### B2. What dropping and filling each cost

*Suggested: 10 minutes.*

Same `survey`. Compute the mean and standard deviation of `income` three ways:

- `dropped`---a tuple `(mean, sd)` over the respondents who answered;
- `filled`---a tuple `(mean, sd)` after filling the gaps with the mean of the
  answers;
- `truth`---a tuple `(mean, sd)` of the incomes *before* any were hidden, which
  the setup keeps in `income_true`.

Predict, before you run it, which of the three standard deviations is smallest.

In [ ]:
income_true = pd.Series(rng.lognormal(10.8, 0.5, n).round(-2))
refuse = rng.random(n) < (income_true / income_true.max()) * 0.6
income = income_true.where(~refuse)

In [ ]:
dropped = ...
filled = ...
truth = ...

for name, pair in [("dropped", dropped), ("filled", filled), ("truth", truth)]:
    print(f"{name:8} mean {pair[0]:>10,.0f}   sd {pair[1]:>10,.0f}")

In [ ]:
# self-check: run this after your answer above
wd = (float(income.dropna().mean()), float(income.dropna().std()))
wf_s = income.fillna(income.mean())
wf = (float(wf_s.mean()), float(wf_s.std()))
wt = (float(income_true.mean()), float(income_true.std()))
for name, got, want in [("dropped", dropped, wd), ("filled", filled, wf), ("truth", truth, wt)]:
    assert len(got) == 2, f"{name} should be a (mean, sd) pair"
    assert abs(float(got[0]) - want[0]) < 1e-6, f"{name}: the mean is not right"
    assert abs(float(got[1]) - want[1]) < 1e-6, f"{name}: the standard deviation is not right"
print("looks right")

### B3. Duplicate by whose definition?

*Suggested: 8 minutes.*

`log` is a click log. One row is a byte-for-byte repeat of another; two more
record the same user and page at different times.

- `n_exact`---how many rows are duplicates under *every* column;
- `n_user_page`---how many are duplicates considering only `user` and `page`;
- `first_kept`---the frame with `user`/`page` duplicates dropped, keeping the
  **first** occurrence;
- `last_kept`---the same, keeping the **last**.

In [ ]:
log = pd.DataFrame({
    "user": ["a", "b", "a", "c", "b", "a"],
    "page": ["/home", "/cart", "/home", "/home", "/cart", "/help"],
    "secs": [12, 30, 12, 8, 44, 19],
})

In [ ]:
n_exact = ...
n_user_page = ...
first_kept = ...
last_kept = ...

print(n_exact, n_user_page, len(first_kept), len(last_kept))
last_kept

In [ ]:
# self-check: run this after your answer above
assert int(n_exact) == int(log.duplicated().sum()), \
    f"n_exact should be {int(log.duplicated().sum())}: rows identical in every column"
assert int(n_user_page) == int(log.duplicated(["user", "page"]).sum()), \
    f"n_user_page should be {int(log.duplicated(['user', 'page']).sum())}"
assert first_kept.equals(log.drop_duplicates(["user", "page"])), \
    "first_kept should drop user/page duplicates, keeping the first"
assert last_kept.equals(log.drop_duplicates(["user", "page"], keep="last")), \
    "last_kept should keep the last occurrence instead"
assert int(first_kept.loc[first_kept["user"] == "b", "secs"].iloc[0]) == 30, \
    "keeping the first should leave b's 30-second visit"
print("looks right")

### B4. The join that doubled the revenue

*Suggested: 12 minutes.*

`orders` has one row per order. `lookup` should have one row per customer, and
does not: one customer appears twice.

- `n_before`---the number of orders;
- `merged`---`orders` merged with `lookup` on `cust_id`;
- `n_after`---the number of rows after the merge;
- `spend_before` and `spend_after`---the total of `total` before and after;
- `offenders`---a list of the `cust_id` values that appear more than once in
  `lookup`.

In [ ]:
orders = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5, 6],
    "cust_id":  ["a", "b", "c", "b", "d", "b"],
    "total":    [30, 55, 12, 80, 20, 45],
})
lookup = pd.DataFrame({
    "cust_id": ["a", "b", "c", "d", "b"],
    "city":    ["Shaw", "Petworth", "Shaw", "Brookland", "Petworth"],
})

In [ ]:
n_before = ...
merged = ...
n_after = ...
spend_before = ...
spend_after = ...
offenders = ...

print(n_before, "->", n_after, " spend", spend_before, "->", spend_after)
print("offenders:", offenders)

In [ ]:
# self-check: run this after your answer above
wm = orders.merge(lookup, on="cust_id")
assert int(n_before) == 6, "n_before is the number of orders"
assert isinstance(merged, pd.DataFrame) and len(merged) == len(wm), \
    f"merged should have {len(wm)} rows: every b order matched two lookup rows"
assert int(n_after) == len(wm), f"n_after should be {len(wm)}"
assert abs(float(spend_before) - float(orders['total'].sum())) < 1e-9, \
    "spend_before is the total before the merge"
assert abs(float(spend_after) - float(wm['total'].sum())) < 1e-9, \
    "spend_after is the total after it"
assert list(offenders) == ["b"], "offenders should be the cust_id values duplicated in lookup"
print("looks right")

### B5. Two levels, and what selecting costs you

*Suggested: 10 minutes.*

`sales` has a row per region, quarter and product line.

- `grouped`---revenue summed by `region` and `quarter`, in that order, as a
  Series with a two-level index;
- `north`---every row of `grouped` for the north region;
- `north_levels`---the number of index levels `north` has;
- `wide`---`grouped` with the quarters moved into the columns.

In [ ]:
sales = pd.DataFrame({
    "region":  ["north", "north", "north", "south", "south", "south", "north", "south"],
    "quarter": ["Q1", "Q1", "Q2", "Q1", "Q2", "Q2", "Q2", "Q1"],
    "line":    ["a", "b", "a", "a", "a", "b", "b", "b"],
    "revenue": [100, 40, 120, 90, 95, 25, 35, 30],
})

In [ ]:
grouped = ...
north = ...
north_levels = ...
wide = ...

print(grouped)
wide

In [ ]:
# self-check: run this after your answer above
wg = sales.groupby(["region", "quarter"])["revenue"].sum()
assert isinstance(grouped, pd.Series), f"grouped should be a Series, got {type(grouped).__name__}"
assert list(grouped.index.names) == ["region", "quarter"], \
    "grouped should be indexed by region then quarter"
assert grouped.equals(wg), "grouped does not match the summed revenue"
assert north.equals(wg.loc["north"]), "north should be every north row of grouped"
assert int(north_levels) == 1, \
    "selecting on the outer level consumes it: north has one level left, not two"
assert isinstance(wide, pd.DataFrame) and list(wide.columns) == ["Q1", "Q2"], \
    "wide should have the quarters as columns"
assert wide.equals(wg.unstack()), "wide should be grouped with the inner level unstacked"
print("looks right")

### B6. Long, wide, and the mean nobody chose

*Suggested: 12 minutes.*

`wide_sales` has a column per month. You need it long for plotting, and then
back to wide for a slide---but the trip back is not symmetric, because the long
frame has two rows for one store-month.

- `long`---`wide_sales` melted, with the month in `month` and the number in
  `sales`;
- `n_long`---its row count;
- `pivot_failed`---`True` if `long.pivot(index="store", columns="month", values="sales")` raises, `False` if it does not;
- `totals`---the same reshape with `pivot_table`, summing the duplicates.

In [ ]:
wide_sales = pd.DataFrame({
    "store": ["A", "B", "A"],
    "jan":   [120, 90, 15],
    "feb":   [135, 88, 20],
})

In [ ]:
long = ...
n_long = ...
pivot_failed = ...
totals = ...

print(long)
totals

In [ ]:
# self-check: run this after your answer above
wl = wide_sales.melt(id_vars="store", var_name="month", value_name="sales")
assert isinstance(long, pd.DataFrame), f"long should be a DataFrame, got {type(long).__name__}"
assert list(long.columns) == ["store", "month", "sales"], \
    "long should have columns store, month, sales"
assert len(long) == len(wl) == 6, f"long should have {len(wl)} rows"
assert bool(pivot_failed) is True, \
    "pivot() should raise here: store A appears twice for each month"
wt = wl.pivot_table(index="store", columns="month", values="sales", aggfunc="sum")
assert isinstance(totals, pd.DataFrame) and totals.equals(wt), \
    "totals should be the pivot_table summed, not averaged"
assert int(totals.loc["A", "jan"]) == 135, "A's January total is 120 + 15"
print("looks right")

### B7. The fill that depends on the sort

*Suggested: 10 minutes.*

`readings` holds a price per product, with gaps. The rows arrive sorted by
`product_id`.

- `naive`---the `price` column after a plain `ffill()`, as a Series;
- `by_category`---the same fill done **within** each category;
- `n_differ`---how many rows the two fills disagree about;
- `still_missing`---how many gaps `by_category` leaves.

In [ ]:
readings = pd.DataFrame({
    "product_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "category":   ["tools", "food", "tools", "food", "tools", "food", "food", "tools"],
    "price":      [10.0, np.nan, np.nan, 3.0, 12.0, np.nan, 4.0, np.nan],
})

In [ ]:
naive = ...
by_category = ...
n_differ = ...
still_missing = ...

print(pd.DataFrame({"category": readings["category"], "price": readings["price"],
                    "naive": naive, "by_category": by_category}))
print(n_differ, "rows differ;", still_missing, "gaps left")

In [ ]:
# self-check: run this after your answer above
wn = readings["price"].ffill()
wc = readings.groupby("category")["price"].ffill()
assert isinstance(naive, pd.Series) and np.allclose(naive.to_numpy(), wn.to_numpy(), equal_nan=True), \
    "naive should be readings['price'].ffill()"
assert isinstance(by_category, pd.Series) and np.allclose(by_category.to_numpy(), wc.to_numpy(), equal_nan=True), \
    "by_category should carry the last price within the same category"
assert int(n_differ) == int((~np.isclose(wn.to_numpy(), wc.to_numpy(), equal_nan=True)).sum()), \
    "n_differ should count the rows where the two fills disagree"
assert int(still_missing) == int(wc.isna().sum()), \
    "still_missing should count the gaps by_category leaves"
print("looks right")

### B8. One number, end to end

*Suggested: 12 minutes.*

No new function. Put the session together on one small pipeline, and watch the
row count at every step.

`tx` is a transaction log with duplicates and gaps; `region_of` maps each store
to a region, and has one store listed twice.

Produce `report`---total `amount` by region, as a Series indexed by region,
sorted from largest to smallest---having:

1. dropped exact duplicate transactions;
2. dropped transactions with no `amount`;
3. merged on `store` **without** letting the duplicated store multiply rows.

Also record `n_steps`---a list of the row count after each of the three steps.

In [ ]:
tx = pd.DataFrame({
    "tx_id":  [1, 2, 3, 3, 4, 5, 6, 7],
    "store":  ["s1", "s2", "s1", "s1", "s3", "s2", "s3", "s1"],
    "amount": [10.0, 25.0, 5.0, 5.0, np.nan, 40.0, 15.0, 20.0],
})
region_of = pd.DataFrame({
    "store":  ["s1", "s2", "s3", "s2"],
    "region": ["north", "south", "north", "south"],
})

In [ ]:
report = ...
n_steps = ...

print(n_steps)
report

In [ ]:
# self-check: run this after your answer above
a = tx.drop_duplicates()
b = a.dropna(subset=["amount"])
c = b.merge(region_of.drop_duplicates("store"), on="store")
want = c.groupby("region")["amount"].sum().sort_values(ascending=False)
assert isinstance(report, pd.Series), f"report should be a Series, got {type(report).__name__}"
assert list(report.index) == list(want.index), \
    "report should be indexed by region, largest total first"
assert np.allclose(report.to_numpy(), want.to_numpy()), \
    "the totals do not match: check that the merge did not duplicate rows"
assert list(map(int, n_steps)) == [len(a), len(b), len(c)], \
    f"n_steps should be the row counts after each step: {[len(a), len(b), len(c)]}"
print("looks right")

## Before you leave

Upload this notebook to Blackboard. It is the record of what you did; the lab
grade itself comes from what you do in the room.

Then post a comment at the bottom of the lab page, <https://www.smajhi.com/DATS-6103/labs/pandas-II.html>. One line is enough.
It proves your GitHub account works with the comment system, which is how every
week's reading credit is earned.